In [1]:
import sys
sys.path.append('../')

import contextlib
import io
import time

import pandas as pd

from utilities import read_instance, read_solution, is_feasible, calculate_cost, score, size_class
from clarke_wright_solution import run_cw_ls_algo, time_budget
from route_first_cluster_second_solution import run_rfcs_algo
from run_benchmarks import find_instances

In [2]:
# Each solver takes (demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit) -> (cost, routes)
SOLVERS = {
    'cw_ls': lambda demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit: run_cw_ls_algo(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=time_limit),
    'rfcs': lambda demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit: run_rfcs_algo(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=time_limit),
}

In [3]:
def run_benchmarks_df(solvers=SOLVERS, sets=('A', 'B', 'P', 'X'), samples_limit=None, time_vehicle_capacity=None):
    '''Runs every solver on the benchmark instances and returns one row per (solver, instance).'''
    rows = []
    for s in sets:

        # limit the number of samples if needed
        instances = list(find_instances([s]))
        if samples_limit:
            instances= instances[:samples_limit]

        for path, opt_path in instances:
            # read the test case and provided solution
            demand, dist_matrix, vehicle_capacity, num_vehicles = read_instance(path)
            optimal_cost = read_solution(opt_path)[0] if opt_path else None

            num_customers = len(demand) - 1
            for name, solver in solvers.items():
                time_start = time.perf_counter()
                _, routes = solver(demand, dist_matrix, vehicle_capacity, num_vehicles, time_limit=None)
                elapsed = time.perf_counter() - time_start

                with contextlib.redirect_stdout(io.StringIO()):   # hide is_feasible's message
                    feasible = is_feasible(routes, demand, vehicle_capacity, num_vehicles)
                cost = calculate_cost(routes, dist_matrix)

                rows.append({
                    # instance variables
                    'solver': name, 'set': s, 'instance': path.stem,
                    'num_customers': num_customers, 'vehicle_capacity': vehicle_capacity, 'num_vehicles': num_vehicles,
                    'total_demandand': sum(demand),
                    # results
                    'time_taken': elapsed, 'cost': cost,
                    'optimal_cost': optimal_cost,
                    'gap_pct': 100 * (cost - optimal_cost) / optimal_cost if optimal_cost else None,
                    'feasible': feasible,
                })
                # print(f'{name:<6} {path.stem:<16} cost={cost:10.1f}  time={elapsed:6.1f}s', flush=True)
    return pd.DataFrame(rows)


# Quick run: first 3 instances of set A, vehicle_capacityped at 5 s each. Drop the limits for the full run.
results = run_benchmarks_df(sets=['A', 'B', 'P'], samples_limit=None, time_vehicle_capacity=5)


In [4]:
results.head()

,solver,set,instance,num_customers,vehicle_capacity,num_vehicles,total_demandand,time_taken,cost,optimal_cost,gap_pct,feasible
0,cw_ls,A,A-n32-k5,31,100,5,410,0.008081,827,784,5.484694,True
1,rfcs,A,A-n32-k5,31,100,5,410,0.180449,784,784,0.000000,True
2,cw_ls,A,A-n33-k5,32,100,5,446,0.002889,702,661,6.202723,True
3,rfcs,A,A-n33-k5,32,100,5,446,0.155866,661,661,0.000000,True
4,cw_ls,A,A-n33-k6,32,100,6,541,0.004015,743,742,0.134771,True


In [5]:
# compares the time taken and lowest cost
results.groupby(['set', 'solver'])[['cost', 'time_taken', 'gap_pct']].mean()

cost  time_taken   gap_pct
set solver                                   
A   cw_ls   1071.148148    0.010609  2.754736
    rfcs    1049.814815    0.354398  0.693300
B   cw_ls    989.782609    0.010959  2.426921
    rfcs     971.347826    0.421773  0.719830
P   cw_ls    618.041667    0.009079  5.569995
    rfcs     596.458333    0.450886  1.316186

In [6]:
results.groupby(['set', 'solver', 'feasible'])['instance'].count()

set  solver  feasible
A    cw_ls   True        27
     rfcs    True        27
B    cw_ls   True        23
     rfcs    True        23
P    cw_ls   True        24
     rfcs    True        24
Name: instance, dtype: int64